# 07 — RAG Knowledge Assistant
## Customer Analytics Platform

Purpose: Build a Retrieval-Augmented Generation (RAG)
system that allows business users to query customer
reviews using natural language.

Architecture:
1. Load customer reviews from Silver Delta table
2. Generate embeddings using sentence-transformers
3. Store embeddings in FAISS vector database
4. Accept natural language business questions
5. Retrieve most relevant reviews semantically
6. Generate business insights using LLM

In [0]:
# install required libraries for RAG pipeline

%pip install sentence-transformers faiss-cpu transformers torch

In [0]:
# restart kernel after installation

dbutils.library.restartPython()

In [0]:
# verify all libraries installed correctly

import sentence_transformers
import faiss
import torch
import transformers

print(f"sentence-transformers : {sentence_transformers.__version__}")
print(f"faiss version         : {faiss.__version__}")
print(f"torch version         : {torch.__version__}")
print(f"transformers version  : {transformers.__version__}")
print("all libraries ready for RAG pipeline")

In [0]:
# load customer reviews from silver delta table

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws

spark = SparkSession.builder.getOrCreate()

silver_path = "/Volumes/workspace/default/olist_raw_data/silver"

reviews_df = spark.read.format("delta") \
    .load(f"{silver_path}/reviews")

# keep only reviews with actual text comments
reviews_with_text = reviews_df \
    .filter(col("review_comment_message").isNotNull()) \
    .filter(col("review_comment_message") != "") \
    .select(
        "order_id",
        "review_score",
        "review_comment_message",
        "has_comment"
    )

print(f"total reviews          : {reviews_df.count():,}")
print(f"reviews with text      : {reviews_with_text.count():,}")
print("\nsample reviews:")
reviews_with_text.show(3, truncate=80)

In [0]:
# convert to pandas and prepare text corpus for embeddings
# using a sample of 10000 reviews for performance

import pandas as pd

reviews_pdf = reviews_with_text \
    .limit(10000) \
    .toPandas()

# clean the text
reviews_pdf["review_text"] = reviews_pdf[
    "review_comment_message"
].str.strip()

# add sentiment label based on score
reviews_pdf["sentiment"] = reviews_pdf["review_score"].apply(
    lambda x: "positive" if x >= 4 else 
              "negative" if x <= 2 else "neutral"
)

print(f"reviews loaded         : {len(reviews_pdf):,}")
print(f"\nsentiment distribution:")
print(reviews_pdf["sentiment"].value_counts())
print(f"\nsample review text:")
print(reviews_pdf["review_text"].iloc[0])

In [0]:
# generate embeddings for all 10000 reviews
# sentence-transformers paraphrase-multilingual handles Portuguese

from sentence_transformers import SentenceTransformer
import numpy as np

print("loading embedding model...")
model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)
print("model loaded successfully")

print("\ngenerating embeddings for 10,000 reviews...")
print("this will take 2-3 minutes...")

review_texts = reviews_pdf["review_text"].tolist()

embeddings = model.encode(
    review_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nembeddings generated!")
print(f"embedding shape : {embeddings.shape}")
print(f"each review     : {embeddings.shape[1]} dimensional vector")

In [0]:
# build FAISS vector index for fast semantic search

import faiss
import numpy as np

# normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# add embeddings to index
index.add(embeddings)

print(f"FAISS index built successfully")
print(f"total vectors indexed : {index.ntotal:,}")
print(f"vector dimension      : {dimension}")
print(f"index type            : IndexFlatIP (cosine similarity)")

In [0]:
# build semantic search function
# takes a business question and finds most relevant reviews

def semantic_search(question, top_k=5):
    # convert question to embedding
    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    )
    
    # normalize for cosine similarity
    faiss.normalize_L2(question_embedding)
    
    # search FAISS index
    scores, indices = index.search(question_embedding, top_k)
    
    # retrieve matching reviews
    results = []
    for score, idx in zip(scores[0], indices[0]):
        review = reviews_pdf.iloc[idx]
        results.append({
            "score"     : round(float(score), 4),
            "sentiment" : review["sentiment"],
            "rating"    : review["review_score"],
            "review"    : review["review_text"]
        })
    
    return results

# test with a business question
question = "Why are customers unhappy with delivery?"
results  = semantic_search(question, top_k=5)

print(f"question: {question}")
print("-" * 60)
for i, r in enumerate(results, 1):
    print(f"\nresult {i}:")
    print(f"  similarity : {r['score']}")
    print(f"  sentiment  : {r['sentiment']}")
    print(f"  rating     : {r['rating']} stars")
    print(f"  review     : {r['review'][:150]}")

In [0]:
# build RAG answer generation using retrieved reviews

def generate_rag_answer(question, top_k=5):
    # step 1 - retrieve relevant reviews
    results = semantic_search(question, top_k)
    
    # step 2 - build context from retrieved reviews
    context = "\n".join([
        f"Review {i+1} ({r['sentiment']}, {r['rating']} stars): {r['review']}"
        for i, r in enumerate(results)
    ])
    
    # step 3 - build prompt for LLM
    prompt = f"""You are a business analyst for an e-commerce platform.
Based on the following customer reviews, answer the business question clearly.
Always respond in English with actionable insights.

Customer Reviews:
{context}

Business Question: {question}

Provide a clear, concise answer based only on the reviews above.
Focus on patterns and actionable business insights."""
    
    return {
        "question"        : question,
        "retrieved_reviews": results,
        "prompt"          : prompt,
        "context_length"  : len(context)
    }

# test it
result = generate_rag_answer(
    "Why are customers unhappy with delivery?"
)

print(f"question       : {result['question']}")
print(f"context length : {result['context_length']} characters")
print(f"\nRAG prompt built successfully")
print(f"\nretrieved {len(result['retrieved_reviews'])} relevant reviews")
print("\nRAG pipeline working end to end!")

In [0]:
# load LLM for answer generation using flan-t5

from transformers import pipeline

print("loading LLM model...")
print("this will take 2-3 minutes...")

generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=200
)

print("LLM loaded successfully!")

In [0]:
# complete RAG pipeline with structured business insight generation

def rag_answer(question):
    # step 1 - retrieve relevant reviews
    results = semantic_search(question, top_k=5)
    
    # step 2 - analyze retrieved reviews
    negative_reviews = [r for r in results if r["sentiment"] == "negative"]
    positive_reviews = [r for r in results if r["sentiment"] == "positive"]
    avg_rating       = sum(r["rating"] for r in results) / len(results)
    
    # step 3 - generate structured business insight
    insight_lines = []
    insight_lines.append(f"Based on {len(results)} most relevant customer reviews:")
    insight_lines.append(f"Average rating of matched reviews: {avg_rating:.1f}/5")
    insight_lines.append("")
    
    if negative_reviews:
        insight_lines.append(f"Negative feedback ({len(negative_reviews)} reviews):")
        for r in negative_reviews[:3]:
            insight_lines.append(f"  - {r['review'][:150]}")
    
    if positive_reviews:
        insight_lines.append(f"\nPositive feedback ({len(positive_reviews)} reviews):")
        for r in positive_reviews[:2]:
            insight_lines.append(f"  - {r['review'][:150]}")
    
    insight_lines.append(f"\nSemantic match score: {results[0]['score']:.4f}")
    
    return {
        "question" : question,
        "insight"  : "\n".join(insight_lines),
        "reviews"  : results
    }

# test with 3 business questions
questions = [
    "Why are customers unhappy with delivery?",
    "What do customers love about the products?",
    "What are the main complaints about product quality?"
]

for q in questions:
    print(f"\nQuestion: {q}")
    print("=" * 60)
    result = rag_answer(q)
    print(result["insight"])
    print()

## RAG Knowledge Assistant Summary

Architecture:
- Embedding Model : paraphrase-multilingual-MiniLM-L12-v2
- Vector Database : FAISS (IndexFlatIP cosine similarity)
- Knowledge Base  : 10,000 Olist customer reviews
- Vector Dimension: 384
- Search Method   : Semantic similarity search

Capabilities:
- Accepts natural language business questions
- Retrieves top 5 most semantically relevant reviews
- Generates structured business insights
- Handles multilingual reviews (Portuguese + English)
- Average semantic match score above 0.70

Business Questions Answered:
- Why are customers unhappy with delivery?
- What do customers love about products?
- What are main complaints about quality?

Key Findings:
- Delivery complaints average 2.6/5 stars
- Product satisfaction averages 4.6/5 stars  
- Quality complaints average 2.4/5 stars
- Main delivery issue: postal service delays
- Main strength: product value for money

Next Steps:
- Connect to Gold layer customer segments
- Add English translation layer
- Deploy as interactive business tool